# Model Refinement for Bike-Sharing Demand Prediction

After establishing a baseline model, the next step is to improve model performance by incorporating more complex relationships between variables.

The baseline model assumes linear relationships between predictors and the target variable. However, real-world data often exhibits:

- Non-linear relationships
- Interactions between variables
- Overfitting risks with increased complexity

In this notebook, we refine the baseline model by:

- Adding polynomial terms
- Introducing interaction effects
- Applying regularization techniques

This aligns with the second stage of regression modeling as outlined in the lab instructions.

## Import Required Libraries

We use:

- `dplyr`: data manipulation
- `ggplot2`: visualization
- `caret`: model evaluation
- `glmnet`: regularization models

In [5]:
# Load required libraries

# dplyr for data manipulation
library(dplyr)

# ggplot2 for visualization
library(ggplot2)

# caret for model evaluation
library(caret)

# glmnet for regularization (Ridge, Lasso, Elastic Net)
library(glmnet)

# Import the readr package which is used for reading the csv files
library(readr) 


Attaching package: 'dplyr'

The following objects are masked from 'package:stats':

    filter, lag

The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union

Loading required package: lattice
Loading required package: Matrix
Loaded glmnet 4.1-10


## Load Dataset and Split Data

We load the cleaned dataset and split it into training and testing sets.

In [6]:
# Load dataset
bike_data <- read_csv("clean_bike_data.csv")

# Set seed for reproducibility
set.seed(123)

# Split dataset (80/20)
train_index <- createDataPartition(bike_data$rented_bike_count, p = 0.8, list = FALSE)

train_data <- bike_data[train_index, ]
test_data <- bike_data[-train_index, ]

Rows: 8465 Columns: 17── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr   (4): seasons, holiday, functioning_day, day_of_week
dbl  (12): rented_bike_count, hour, temperature, humidity, wind_speed, visib...
date  (1): date
ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.

## Add Polynomial Terms

Polynomial terms help capture non-linear relationships between variables.

For example:
- Temperature squared may capture diminishing or accelerating effects

In [7]:
# Add polynomial features

train_data <- train_data %>%
  mutate(
    temperature_sq = temperature^2,
    humidity_sq = humidity^2
  )

test_data <- test_data %>%
  mutate(
    temperature_sq = temperature^2,
    humidity_sq = humidity^2
  )

## Add Interaction Terms

Interaction terms capture combined effects between variables.

For example:
- Temperature × Humidity may influence demand differently than each variable independently

In [8]:
# Add interaction feature

train_data <- train_data %>%
  mutate(temp_humidity = temperature * humidity)

test_data <- test_data %>%
  mutate(temp_humidity = temperature * humidity)

## Train Enhanced Linear Regression Model

In [9]:
# Train refined linear model

refined_model <- lm(
  rented_bike_count ~ temperature + humidity + wind_speed +
    temperature_sq + humidity_sq + temp_humidity,
  data = train_data
)

# View model summary
summary(refined_model)


Call:
lm(formula = rented_bike_count ~ temperature + humidity + wind_speed + 
    temperature_sq + humidity_sq + temp_humidity, data = train_data)

Residuals:
     Min       1Q   Median       3Q      Max 
-1260.53  -289.24   -80.66   197.01  2292.90 

Coefficients:
                 Estimate Std. Error t value Pr(>|t|)    
(Intercept)    -103.02981   47.30070  -2.178   0.0294 *  
temperature      53.61352    1.64561  32.580  < 2e-16 ***
humidity         22.32113    1.53455  14.546  < 2e-16 ***
wind_speed       51.99721    5.97166   8.707  < 2e-16 ***
temperature_sq   -0.44819    0.03980 -11.260  < 2e-16 ***
humidity_sq      -0.24549    0.01295 -18.953  < 2e-16 ***
temp_humidity    -0.21194    0.02741  -7.732 1.22e-14 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Residual standard error: 474 on 6767 degrees of freedom
Multiple R-squared:  0.4566,	Adjusted R-squared:  0.4561 
F-statistic: 947.7 on 6 and 6767 DF,  p-value: < 2.2e-16


## Generate Predictions

In [10]:
# Predict on test dataset
predictions_refined <- predict(refined_model, newdata = test_data)

## Evaluate Refined Model

In [11]:
# RMSE function
rmse <- function(actual, predicted) {
  sqrt(mean((actual - predicted)^2))
}

# Compute RMSE
rmse_refined <- rmse(test_data$rented_bike_count, predictions_refined)

# Compute R-squared
r2_refined <- cor(test_data$rented_bike_count, predictions_refined)^2

# Display metrics
rmse_refined
r2_refined

[1] 472.472

[1] 0.4568859

## Apply Regularization (Ridge, Lasso, Elastic Net)

Regularization helps prevent overfitting by penalizing large coefficients.

We use `glmnet` for:

- Ridge Regression (alpha = 0)
- Lasso Regression (alpha = 1)
- Elastic Net (alpha between 0 and 1)

In [13]:
# --------------------------------------------
# STEP 1: Convert character columns to factors
# --------------------------------------------

# Convert all character columns in training data to factors
train_data[] <- lapply(train_data, function(x) {
  if (is.character(x)) factor(x) else x   # If column is character, convert to factor
})

# Convert all character columns in test data to factors
test_data[] <- lapply(test_data, function(x) {
  if (is.character(x)) factor(x) else x   # Apply same transformation to test data
})

# --------------------------------------------
# STEP 2: Identify problematic factor columns
# --------------------------------------------

# Detect factor columns that have only ONE unique value in training data
problematic_cols <- sapply(train_data, function(x) {
  is.factor(x) && length(unique(x)) < 2   # TRUE if factor has < 2 unique values
})

# Print names of problematic columns for debugging
cat("Columns with only one unique value:\n")
print(names(train_data)[problematic_cols])

# --------------------------------------------
# STEP 3: Remove problematic columns
# --------------------------------------------

# Remove identified problematic columns from training dataset
train_data_filtered <- train_data[, !problematic_cols]

# Remove same columns from test dataset to keep structure consistent
test_data_filtered  <- test_data[, !problematic_cols]

# --------------------------------------------
# STEP 4: Align factor levels between train and test
# --------------------------------------------

# Loop through each column in filtered training data
for (col in names(train_data_filtered)) {
  
  # Check if column is a factor
  if (is.factor(train_data_filtered[[col]])) {
    
    # Force test data to use same factor levels as training data
    test_data_filtered[[col]] <- factor(
      test_data_filtered[[col]],
      levels = levels(train_data_filtered[[col]])
    )
  }
}

# --------------------------------------------
# STEP 5: FINAL SAFETY CHECK (IMPORTANT)
# --------------------------------------------

# Loop again to ensure no factor still has only one unique value
for (col in names(train_data_filtered)) {
  
  # Check if column is a factor
  if (is.factor(train_data_filtered[[col]])) {
    
    # If still only one unique value, remove it
    if (length(unique(train_data_filtered[[col]])) < 2) {
      
      # Print column name being removed
      cat("Removed column:", col, "\n")
      
      # Remove column from both datasets
      train_data_filtered[[col]] <- NULL
      test_data_filtered[[col]]  <- NULL
    }
  }
}

# --------------------------------------------
# STEP 6: Create model matrices
# --------------------------------------------

# Convert training dataset into model matrix (one-hot encoding applied automatically)
x_train <- model.matrix(rented_bike_count ~ ., train_data_filtered)[, -1]  # Remove intercept column

# Extract target variable from training dataset
y_train <- train_data_filtered$rented_bike_count

# Convert test dataset into model matrix using same structure
x_test <- model.matrix(rented_bike_count ~ ., test_data_filtered)[, -1]

# --------------------------------------------
# STEP 7: Train models using glmnet
# --------------------------------------------

# Load glmnet library for regularized regression models
library(glmnet)

# Train Ridge Regression model (alpha = 0)
ridge_model <- cv.glmnet(x_train, y_train, alpha = 0)

# Train Lasso Regression model (alpha = 1)
lasso_model <- cv.glmnet(x_train, y_train, alpha = 1)

# Train Elastic Net model (alpha = 0.5)
elastic_model <- cv.glmnet(x_train, y_train, alpha = 0.5)

Columns with only one unique value:[1] "functioning_day"

## Predictions with Regularized Models

In [14]:
# Predict using best lambda

ridge_pred <- predict(ridge_model, s = "lambda.min", newx = x_test)
lasso_pred <- predict(lasso_model, s = "lambda.min", newx = x_test)
elastic_pred <- predict(elastic_model, s = "lambda.min", newx = x_test)

In [15]:
# R² for ridge / lasso / elastic net
# Note: predict(cv.glmnet, ...) returns a matrix — flatten with as.numeric
r2_ridge   <- cor(test_data$rented_bike_count, as.numeric(ridge_pred  ))^2
r2_lasso   <- cor(test_data$rented_bike_count, as.numeric(lasso_pred  ))^2
r2_elastic <- cor(test_data$rented_bike_count, as.numeric(elastic_pred))^2

c(r2_ridge = r2_ridge, r2_lasso = r2_lasso, r2_elastic = r2_elastic)

r2_ridge   r2_lasso r2_elastic 
 0.5610522  0.5795669  0.5796128

## Compare Model Performance

In [27]:
# Calculate RMSE for all models

rmse_ridge <- rmse(test_data$rented_bike_count, ridge_pred)
rmse_lasso <- rmse(test_data$rented_bike_count, lasso_pred)
rmse_elastic <- rmse(test_data$rented_bike_count, elastic_pred)

# Display results
rmse_ridge
rmse_lasso
rmse_elastic

[1] 425.1255

[1] 415.6589

[1] 415.6351

## Interpretation

The performance metrics obtained from the refined models indicate the following:

- The Ridge model produced an error value of approximately 425.13, showing stable performance but slightly higher prediction error due to uniform coefficient shrinkage.
- The Lasso model reduced the error to approximately 415.66, suggesting that feature selection through coefficient shrinkage improved model efficiency.
- The Elastic Net model achieved the lowest error at approximately 415.64, indicating the best overall performance among the three models.

These results demonstrate that incorporating both regularization and feature selection enhances predictive accuracy. The Elastic Net model, which combines the strengths of Ridge and Lasso, provides the most balanced and effective solution for this dataset.

---

## Author & Acknowledgment

**Author:**  
<span style="color:blue">Deepan Mehta </span>  

**GitHub Profile:**  
https://github.com/deepan-mehta-analytics

This notebook focuses on improving the baseline regression model by introducing polynomial features, interaction terms, and regularization techniques.

The workflow follows <span style="color:blue">IBM Skills Network </span> instructional labs on regression model refinement and performance optimization.

Special acknowledgment is given to:

- <span style="color:blue">Yan Luo </span>  
- <span style="color:blue">Jeff Grossman</span>  

---

## Project Context

This notebook represents the model refinement stage in the end-to-end data science pipeline:

- Data Collection  
- Data Wrangling (ETL)  
- Exploratory Data Analysis (EDA)  
- Baseline Model Development  
- **Model Refinement**  
- Model Evaluation  
- Deployment (R Shiny Dashboard)

---

## Notes

The refined models demonstrate improved predictive capability by capturing non-linear relationships and reducing overfitting through regularization.

---